In [1]:
!pip install -q vllm
!pip uninstall -y torchaudio

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ydata-profiling 4.18.4 requires numba<0.63,>=0.60, but you have numba 0.65.0 which is incompatible.
numba-cuda 0.28.2 requires cuda-core<1.0.0,>=0.5.1, but you have cuda-core 1.2.2 which is incompatible.
cuml-cu13 26.6.0 requires numba<0.65.0,>=0.60.0, but you have numba 0.65.0 which is incompatible.
cudf-cu13 26.6.0 requires numba<0.65.0,>=0.60.0, but you have numba 0.65.0 which is incompatible.
gradio 6.30.0 requires pillow<13.0,>=8.0, but you have pillow 13.0.0 which is incompatible.
google-adk 2.7.1 requires opentelemetry-api<=1.42.1,>=1.39, but you have opentelemetry-api 1.45.2 which is incompatible.
google-adk 2.7.1 requires opentelemetry-sdk<=1.42.1,>=1.39, but you have opentelemetry-sdk 1.45.2 which is incompatible.
Found existing installation: torchaudio 2.11.0+cu130
Uninstalling torchaudio-2.11.0+cu130:


In [2]:
!git clone https://github.com/VatsalRoy/keep-it-warm.git
%cd /kaggle/working/keep-it-warm
!python -m env.generate_world --seed 42
!ls runs/baseline_sessions

Cloning into 'keep-it-warm'...
remote: Enumerating objects: 153, done.
remote: Counting objects: 100% (153/153), done.
remote: Compressing objects: 100% (111/111), done.
remote: Total 153 (delta 58), reused 122 (delta 30), pack-reused 0 (from 0)
Receiving objects: 100% (153/153), 163.48 KiB | 3.71 MiB/s, done.
Resolving deltas: 100% (58/58), done.
/kaggle/working/keep-it-warm
seed=42  world_sha256=4a5b9a1a8c73a5e39f83a7c00eb5d4ccafec0d2bf6912ce82e81f31f2f0dc419
episodes.jsonl	turns.jsonl


In [3]:
import os, sys, subprocess, time, requests

def start_server(gpu, port):
    cmd = ["vllm", "serve", "Qwen/Qwen3-4B-Instruct-2507", "--dtype", "half",
           "--max-model-len", "28672", "--enable-prefix-caching", "--enable-prompt-tokens-details",
           "--gpu-memory-utilization", "0.90", "--port", str(port)]
    env = {**os.environ, "CUDA_VISIBLE_DEVICES": str(gpu), "VLLM_SERVER_DEV_MODE": "1"}
    return subprocess.Popen(cmd, stdout=open(f"vllm_gpu{gpu}.log", "w"), stderr=subprocess.STDOUT, env=env)

def wait_ready(proc, port, log):
    for i in range(120):
        if proc.poll() is not None:
            print(f"port {port} CRASHED"); os.system(f"tail -n 30 {log}"); return False
        try:
            if requests.get(f"http://localhost:{port}/health", timeout=2).status_code == 200:
                print(f"port {port} READY after ~{i * 10}s"); return True
        except Exception:
            pass
        time.sleep(10)
    print(f"port {port} not ready after 20 min"); return False

s0 = start_server(0, 8000); assert wait_ready(s0, 8000, "vllm_gpu0.log")
s1 = start_server(1, 8001); assert wait_ready(s1, 8001, "vllm_gpu1.log")
for port in (8000, 8001):
    print(port, "reset ->", requests.post(f"http://localhost:{port}/reset_prefix_cache").status_code)

port 8000 READY after ~230s
port 8001 READY after ~90s
8000 reset -> 200
8001 reset -> 200


In [4]:
!python scripts/run_episodes.py --policy window --sessions s01 --out runs/smoke/window --base http://localhost:8000
!python scripts/run_episodes.py --policy summary --sessions s09 --out runs/smoke/summary --base http://localhost:8001
!echo "summary calls:" && grep -c '"action": "summarize"' runs/smoke/summary/turns.jsonl
!python scripts/summarize_run.py runs/smoke/window | grep -E "prefix edits|unparseable"

s01            s0 4/4 ✓✓✓✓  turns=18  end=completed        peak_ctx=4399   cached=44% $B=0.1631 wall=102.5s
s09            s0 2/4 ✗✗✓✓  turns=27  end=completed        peak_ctx=10684  cached=90% $B=0.1465 wall=158.1s
summary calls:
1
prefix edits (first_changed < previous length): 11 of 17 turns
unparseable replies: 0


In [5]:
jobs = {g: subprocess.Popen([sys.executable, "scripts/run_matrix.py", "--group", g, "--base", f"http://localhost:{p}"],
                            stdout=open(f"matrix_{g}.log", "w"), stderr=subprocess.STDOUT)
        for g, p in [("gpu0", 8000), ("gpu1", 8001)]}
t0 = time.time()
while any(j.poll() is None for j in jobs.values()):
    time.sleep(300)
    print(f"--- {(time.time() - t0) / 60:.0f} min ---", flush=True)
    for g in jobs:
        os.system(f"grep -E 'START|DONE|FAILED' matrix_{g}.log | tail -n 1; tail -n 1 matrix_{g}.log")
print("ALL DONE after", round((time.time() - t0) / 60), "min")

--- 5 min ---
[gpu0] START P1_window6: --policy window --k 6
[gpu0] START P1_window6: --policy window --k 6
[gpu1] START P2_mask4: --policy mask --k 4
[gpu1] START P2_mask4: --policy mask --k 4
--- 10 min ---
[gpu0] START P1_window6: --policy window --k 6
[gpu0] START P1_window6: --policy window --k 6
[gpu1] START P2_mask4: --policy mask --k 4
[gpu1] START P2_mask4: --policy mask --k 4
--- 15 min ---
[gpu0] START P1_window6: --policy window --k 6
[gpu0] START P1_window6: --policy window --k 6
[gpu1] START P2_mask4: --policy mask --k 4
[gpu1] START P2_mask4: --policy mask --k 4
--- 20 min ---
[gpu0] START P1_window6: --policy window --k 6
[gpu0] START P1_window6: --policy window --k 6
[gpu1] START P2_mask4: --policy mask --k 4
[gpu1] START P2_mask4: --policy mask --k 4
--- 25 min ---
[gpu0] START P1_window6: --policy window --k 6
[gpu0] START P1_window6: --policy window --k 6
[gpu1] START P2_mask4: --policy mask --k 4
[gpu1] START P2_mask4: --policy mask --k 4
--- 30 min ---
[gpu0] STAR

In [6]:
!mkdir -p runs/main/P0_full && cp runs/baseline_sessions/*.jsonl runs/main/P0_full/
!grep -h -E "DONE|FAILED" matrix_gpu0.log matrix_gpu1.log
!python scripts/compare_runs.py runs/main/P0_full runs/main/P1_window6 runs/main/P2_mask4 runs/main/P3_batch6 runs/main/P4_summary8k
!python scripts/compare_runs.py --sessions s01,s05,s07,s12 runs/main/P0_full runs/main/P5_timestamp
!python scripts/compare_runs.py --sessions s02,s03,s08,s10 runs/main/P0_full runs/main/P3_batch3 runs/main/P3_batch6 runs/main/P3_batch10
!zip -qr /kaggle/working/results_main.zip runs/main runs/smoke matrix_gpu0.log matrix_gpu1.log vllm_gpu0.log vllm_gpu1.log
!ls -la /kaggle/working/results_main.zip

[gpu0] DONE P1_window6 in 47.2 min
[gpu0] DONE P4_summary8k in 32.0 min
[gpu0] DONE P5_timestamp in 27.1 min
[gpu1] DONE P2_mask4 in 52.6 min
[gpu1] DONE P3_batch6 in 40.0 min
[gpu1] DONE P3_batch3 in 29.1 min
[gpu1] DONE P3_batch10 in 16.5 min
policy         sess   acc turns    peak    tokens cached  naive$B      $A      $B  tokens naive$B  true$A  true$B GPU min
                                                                                        --- change vs P0_full ---
P0_full          12   44%   276  11,132 1,679,446    93%    8.922   0.147   1.886     +0%     +0%     +0%     +0%    32.6
P1_window6       12   38%   295   6,226   909,557    37%    5.215   0.127   3.698    -46%    -42%    -14%    +96%    47.2
P2_mask4         12   31%   337   6,994 1,285,752    61%    7.080   0.149   3.526    -23%    -21%     +1%    +87%    52.5
P3_batch6        12   35%   305   7,695 1,314,809    84%    7.269   0.131   2.305    -22%    -19%    -11%    +22%    40.0
P4_summary8k     12   35%   282